# Figures for Canine Sarcomas Data
Dataset can be downloaded from https://www.ebi.ac.uk/pride/archive/projects/PXD010990 and paper from https://doi.org/10.1016/j.ccell.2018.09.009.

## Preparation

In [ ]:
import os,csv,random
import pandas as pd
import numpy as np
import scanpy as sc
import math

from skimage import io, color
import torch

In [ ]:
from scanpy import read_10x_h5
import SpaGCN as spg
import matplotlib.pyplot as plt
import json
from tqdm import tqdm
import pickle

In [ ]:
from pyimzml.ImzMLParser import ImzMLParser
from pyimzml.metadata import Metadata

In [ ]:
# === CONFIGURE ME ===
DATA_DIR = 'data'        # input data root (subdirectories per dataset live underneath)
OUTPUT_DIR = 'output_sarcomas'    # where this notebook writes its outputs
REPO_ROOT = '.'                # any remaining absolute-path references resolve to here
# ====================


In [ ]:
import GalaxyPython as gx
gx.__version__

## Reading Data
import Ds1, (2), 3, 4, 18, 19, 20, 24, (26) (normal)

In [ ]:
file_loc = f'{DATA_DIR}/CanineSarcomas'

In [ ]:
MALDIdataAnn_cancer = sc.read_h5ad(filename  = file_loc + "/CanineSarcomas_Ds2_05.h5ad") # change here
MALDIdataAnn_normal = sc.read_h5ad(filename  = file_loc + "/CanineSarcomas_Ds26_05.h5ad")

In [ ]:
MALDIdataAnn_cancer = sc.read_h5ad(filename  = file_loc + "/CanineSarcomas_Ds2_05.h5ad") # change here
MALDIdataAnn_normal = sc.read_h5ad(filename  = file_loc + "/CanineSarcomas_Ds26_05.h5ad")

MALDIdataAnn_cancer.obs = MALDIdataAnn_cancer.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn_cancer)

MALDIdataAnn_normal.obs = MALDIdataAnn_normal.obs.astype(int)
sc.pp.normalize_per_cell(MALDIdataAnn_normal)

## Plots of m/z shift

In [ ]:
PeakGroup = gx.PeakCalling(MALDIdataAnn_cancer, MALDIdataAnn_normal)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn_cancer, MALDIdataAnn_normal)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)

In [ ]:
ExactAlign.summarize()

In [ ]:
aligned_index_normal = ExactAlign.referenalign

In [ ]:
aligned_index_cancer = ExactAlign.unknownalign

In [ ]:
shift = []

for i in tqdm(range(len(MALDIdataAnn_normal.var))):
    if i in aligned_index_normal:
        index_aligned = int(np.where(aligned_index_normal == i)[0]) # find the index of i in 'aligned_index_cancer'
        index = aligned_index_cancer[index_aligned] # find the aligned cancer index in mz values
        d = float(MALDIdataAnn_cancer.var_names[index]) - float(MALDIdataAnn_normal.var_names[i])
        shift.append(d)
    else:
        shift.append(0.0)

In [ ]:
shift_dic = {'mzvalues': MALDIdataAnn_cancer.var_names,
         'shift': shift}
df_shift = pd.DataFrame(shift_dic)
df_shift.reset_index(drop=True, inplace=True)

In [ ]:
csv_file_path = f'{OUTPUT_DIR}'

In [ ]:
df_shift.to_csv(csv_file_path + '/shift_ds2_par3.csv', index = False)

In [ ]:
plt.plot(MALDIdataAnn_normal.var_names, shift, linewidth = 1, label = 'Difference', color = 'black')
plt.xlabel('m/z values')
plt.ylabel('shift')
plt.xticks(np.arange(0, 9000, 1500))
plt.yticks([-1, 0, 1])
plt.title('Ds2')

### Using code from MALDI package

In [ ]:
PeakGroup = gx.PeakCalling(MALDIdataAnn_cancer, MALDIdataAnn_normal)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn_cancer, MALDIdataAnn_normal)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)
ExactAlign.summarize()

In [ ]:
## Shifting distribution plot
SD = gx.MALDI_SIM(ExactAlign)
file_loc = f'{REPO_ROOT}/'
SD.shiftdatadf.to_csv("output/new_shift/mzshifting_Ds2_par3.csv", sep = ",")

## Plots of Pearson's Coefficient

In [ ]:
import seaborn as sns

In [ ]:
PeakGroup = gx.PeakCalling(MALDIdataAnn_cancer, MALDIdataAnn_normal)
PeakGroup.peak_calling(0.9)
PeakGroup.peak_grouping(0.9)

In [ ]:
sum(PeakGroup.peakRef)

In [ ]:
sum(PeakGroup.peakUnk)

In [ ]:
ExactAlign = gx.AnnDataMALDI(MALDIdataAnn_cancer, MALDIdataAnn_normal)
ExactAlign.get_corr_peakgroup_refined(PeakGroup.jointcluster)
ExactAlign.peak_group_pairing()
ExactAlign.fine_alignment_assessment(threshold= 0.2, ignore = True)

In [ ]:
len(PeakGroup.jointcluster)

In [ ]:
# PeakGroup.jointcluster

In [ ]:
ExactAlign.PearsonMatrixFull

In [ ]:
num_rows, num_columns = ExactAlign.PearsonMatrixFull.shape
print(f"Number of lists (rows): {num_rows}")
print(f"Number of elements in each list (columns): {num_columns}")
# There are 99 clusters, slide each cluster from -4 to 4.

In [ ]:
len(ExactAlign.PearsonMatrixFull)

In [ ]:
ExactAlign.nclusters

In [ ]:
# np.apply_along_axis(len, axis = 1, arr = ExactAlign.PearsonMatrixFull)

In [ ]:
sns.heatmap(ExactAlign.PearsonMatrixFull, cmap = 'RdBu', square = True)
plt.show()

In [ ]:
sns.heatmap(ExactAlign.PearsonMatrixFull[850:900, 850:900], cmap = 'RdBu', square = True)
plt.show()
# 850 - 900, top left

In [ ]:
PeakGroup.jointcluster[94]

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[94]]

In [ ]:
PeakGroup.jointcluster[7]

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[7]]

In [ ]:
# ExactAlign.mz_valueRef.iloc[ExactAlign.aligned_mz_clusters_ref[43]]

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Define RGB values (0 to 255) for the red color
red_rgb = (150, 5, 5)

# Convert RGB values to the range expected by Matplotlib (0 to 1)
red_color = tuple(component / 255.0 for component in red_rgb)

# Define a custom colormap
colors = [red_color, 
          (1, 1, 1), 
          red_color]  # Red to white to red
custom_cmap = LinearSegmentedColormap.from_list('custom_red_white_red', colors, N=256)

In [ ]:
y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4']
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

sns.heatmap(ExactAlign.PearsonMatrixFull[855:864, 855:864], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1)

plt.xlabel('Reference', fontsize = 14)
plt.ylabel('Unknown', fontsize = 14)
# 850 - 900, top left

# Create a y = -x line
x_values = np.arange(len(x_labels) + 1)
y_values = x_values
sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)
plt.suptitle("group 96 vs. group 96", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_grp96.png', dpi=300)  # Adjust the filename and dpi as needed
plt.show()

In [ ]:
sns.heatmap(ExactAlign.PearsonMatrixFull[50:100, 50:100], cmap = 'RdBu', square = True)
plt.show()
# 50 - 100: top left

In [ ]:
y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4']
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

sns.heatmap(ExactAlign.PearsonMatrixFull[54:63, 54:63], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1)

plt.xlabel('Reference', fontsize = 14)
plt.ylabel('Unknown', fontsize = 14)

sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)
plt.suptitle("group 7 vs. group 7", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_grp7.png', dpi=300)
plt.show()
# 50 - 100: top left

In [ ]:
sns.heatmap(ExactAlign.PearsonMatrixFull[10:60, 10:60], cmap = 'RdBu', square = True)
plt.show()
# 10 - 60: left bottom

In [ ]:
y_labels = ['U-4', 'U-3', 'U-2', 'U-1', 'U0', 'U1', 'U2', 'U3', 'U4']
x_labels = ['R-4', 'R-3', 'R-2', 'R-1', 'R0', 'R1', 'R2', 'R3', 'R4']

sns.heatmap(ExactAlign.PearsonMatrixFull[36:45, 27:36], cmap = 'RdBu_r', square = True,
            xticklabels = x_labels, yticklabels = y_labels, vmin = -1, vmax = 1)

plt.xlabel('Reference', fontsize = 14)
plt.ylabel('Unknown', fontsize = 14)

sns.lineplot(x = x_values, y = y_values, color = 'black', linestyle = '-', linewidth = 1)
plt.suptitle("group 5 vs. group 4", fontsize = 18, y = 0.95, x = 0.455)
plt.savefig('heatmap_grp5.png', dpi=300)
plt.show()
# 10 - 60: left bottom